# ☀️ VoxCity 일사량 분석

EPW(EnergyPlus Weather) 데이터와 복셀 도시 모델을 통한 3D 레이 트레이싱(Ray-tracing)을 사용하여 **순간** 및 **누적** 일사량을 계산합니다.

## 학습 내용

| 분석 유형 | 설명 |
|---------------|-------------|
| **순간 일사량** | 특정 날짜/시간의 일사량 |
| **누적 일사량** | 특정 기간 동안의 전체 일사량 합계 |

## 주요 기능

- 위치 기반 자동 EPW 기상 파일 다운로드
- 법선면 직달 일사량(DNI) 및 수평면 확산 일사량(DHI) 모델링
- 3D 복셀 기하구조를 통한 그림자 생성
- 수관(Canopy) 투과율 반영
- 시각화를 위한 색상화된 OBJ 메쉬 내보내기

## 사전 요구 사항

```python
pip install voxcity
```

In [ ]:
# 설치 (새로운 환경에서 실행하는 경우 주석을 해제하세요)
# %pip install voxcity
# %pip install contextily osmnx geopandas matplotlib timezonefinder


In [ ]:
import ee
from voxcity.geoprocessor.draw import draw_rectangle_map_cityname, center_location_map_cityname
from voxcity.generator import get_voxcity
from voxcity.simulator.solar import get_global_solar_irradiance_using_epw

# Earth Engine 인증 및 초기화 (파이프라인에서 GEE를 통해 데이터를 다운로드할 경우에만 필요합니다)
# ee.Authenticate()
# ee.Initialize(project='your-project-id')

cityname = "Tokyo, Japan"
meshsize = 5

# 옵션 A: 도시 주변에 대화형으로 사각형 그리기
# m, rectangle_vertices = draw_rectangle_map_cityname(cityname, zoom=15)
# m  # 지도를 표시하고 사각형을 그린 뒤 rectangle_vertices를 캡처합니다

# 옵션 B: 클릭하여 고정된 너비/높이(m)로 중심점 설정
# m, rectangle_vertices = center_location_map_cityname(cityname, east_west_length=500, north_south_length=500, zoom=15)
# m

# 옵션 C: 알고 있는 경우 사각형 좌표(경도, 위도) 직접 제공
rectangle_vertices = [
    (139.760, 35.680),  # 남서(SW)
    (139.760, 35.690),  # 북서(NW)
    (139.770, 35.690),  # 북동(NE)
    (139.770, 35.680)   # 남동(SE)
]


In [ ]:
building_source = 'OpenStreetMap'
land_cover_source = 'OpenStreetMap'
canopy_height_source = 'High Resolution 1m Global Canopy Height Maps'
dem_source = 'DeltaDTM'

kwargs = {
    "output_dir": "output/solar_demo",
    # 내부적으로 GeoTIFF 소스를 읽을 때 선택적인 DEM 평활화 적용
    "dem_interpolation": True,
}

city = get_voxcity(
    rectangle_vertices,
    meshsize=meshsize,
    building_source=building_source,
    land_cover_source=land_cover_source,
    canopy_height_source=canopy_height_source,
    dem_source=dem_source,
    **kwargs
)

# VoxCity 객체에서 그리드 데이터에 접근
voxcity_grid = city.voxels.classes
dem_grid = city.dem.elevation

print(voxcity_grid.shape, dem_grid.shape)


---
## ⚡ 순간 일사량

특정 **날짜와 시간**의 전체 일사량을 계산합니다.

### 주요 매개변수

| 매개변수 | 설명 |
|-----------|-------------|
| `calc_time` | 날짜 및 시간 (형식: "MM-DD HH:MM:SS") |
| `download_nearest_epw` | 가장 가까운 기상 파일 자동 다운로드 여부 |
| `tree_k` | 나무의 소멸 계수 |
| `tree_lad` | 엽면적 밀도(LAD) |
| `direct_normal_irradiance_scaling` | DNI 보정 계수 |
| `diffuse_irradiance_scaling` | DHI 보정 계수 |

In [ ]:
solar_kwargs = {
    "download_nearest_epw": True,
    "rectangle_vertices": rectangle_vertices,
    # 또는 로컬 EPW 파일을 직접 설정:
    # "epw_file_path": "./output/your_city.epw",
    "calc_time": "01-01 12:00:00",
    "view_point_height": 1.5,
    "tree_k": 0.6,
    "tree_lad": 1.0,
    "colormap": "magma",
    "obj_export": True,
    "output_directory": "output/solar_demo",
    "output_file_name": "instantaneous_solar_irradiance",
    "alpha": 1.0,
    "vmin": 0,
    # "vmax": 900,
}

solar_grid = get_global_solar_irradiance_using_epw(
    city,
    calc_type='instantaneous',
    direct_normal_irradiance_scaling=1.0,
    diffuse_irradiance_scaling=1.0,
    **solar_kwargs
)

solar_grid.shape


---
## 📊 누적 일사량

특정 기간(예: 일별, 월별, 계절별) 동안의 **전체 일사량**을 계산합니다.

### 주요 매개변수

| 매개변수 | 설명 |
|-----------|-------------|
| `start_time` | 누적 시작 시간 (MM-DD HH:MM:SS) |
| `end_time` | 누적 종료 시간 (MM-DD HH:MM:SS) |
| `time_step_minutes` | 적분 시간 간격(분 단위) |
| `start_hour` / `end_hour` | 특정 시간대로 계산 제한 (예: 오전 8시 ~ 오후 6시) |

**참고:** 누적 계산은 구역이 넓거나 기간이 길 경우 연산량이 매우 많을 수 있습니다.

In [ ]:
cum_kwargs = solar_kwargs.copy()
# 누적 계산을 위한 시간 범위 정의
cum_kwargs["start_time"] = "01-01 05:00:00"
cum_kwargs["end_time"] = "01-31 20:00:00"

# 선택적으로 매일 특정 시간대만 제한 (예: 오전 8시 ~ 오후 4시)
cum_kwargs["start_hour"] = 8
cum_kwargs["end_hour"] = 16

# 성능 제어 (내부 numba 스레드 수)
cum_kwargs["numba_num_threads"] = 4
cum_kwargs["progress_report"] = True

cum_kwargs["output_file_name"] = "cumulative_solar_irradiance"

cum_solar_grid = get_global_solar_irradiance_using_epw(
    city,
    calc_type='cumulative',
    direct_normal_irradiance_scaling=1.0,
    diffuse_irradiance_scaling=1.0,
    **cum_kwargs
)

cum_solar_grid.shape


## 팁
- SSL 문제로 EPW 다운로드가 실패할 경우, kwargs에 `allow_insecure_ssl=True` 또는 `allow_http_fallback=True` 설정을 고려하세요.
- 반복적인 다운로드를 피하려면 `download_nearest_epw=False`로 설정하고 `epw_file_path`를 직접 제공하세요.
- 식생 투과율을 조절하려면 `tree_k`와 `tree_lad` 값을 조정하세요.
